In [7]:
# Importación de librerías
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_exoplanets= pd.read_csv("exoplanets.csv", low_memory=False)

In [8]:
#Objetivo del proyecto. 
#Preguntas de investigación
#Data set: Exo-planetary systems discovered by NASA download from Kaggle - Sathyanarayan Rao

#Importacion de csv a df, usar relative path
df = pd.read_csv("exoplanets.csv", low_memory=False)

In [9]:
master_columns = [
    "pl_name",
    "hostname",
    "discoverymethod",
    "disc_year",
    "disc_facility",
    "default_flag",
    "soltype",
    "pl_controv_flag",
    "pl_orbper",
    "pl_orbsmax",
    "pl_rade",
    "pl_bmasse",
    "pl_orbeccen",
    "pl_eqt",
    "st_mass",
    "st_rad",
    "st_teff",
    "sy_snum",
    "sy_pnum"
]

# Crear dataset de trabajo
df_exoplanets = df[master_columns].copy()

# Conservar únicamente la solución predeterminada
df_exoplanets = df_exoplanets[
    df_exoplanets["default_flag"] == 1
].copy()

# Revisar dimensiones de data set (19 col x 5044 rows)
print("Dimensiones:", df_exoplanets.shape)

# Revisar 
df_exoplanets.head(5)

Dimensiones: (5044, 19)


,pl_name,hostname,discoverymethod,disc_year,disc_facility,default_flag,soltype,pl_controv_flag,pl_orbper,pl_orbsmax,pl_rade,pl_bmasse,pl_orbeccen,pl_eqt,st_mass,st_rad,st_teff,sy_snum,sy_pnum
0,11 Com b,11 Com,Radial Velocity,2007,Xinglong Station,1,Published Confirmed,0,326.03000,1.29,NaN,6165.6000,0.231,NaN,2.70,19.00,4742.0,2,1
3,11 UMi b,11 UMi,Radial Velocity,2009,Thueringer Landessternwarte Tautenburg,1,Published Confirmed,0,516.21997,1.53,NaN,4684.8142,0.080,NaN,2.78,29.79,4213.0,1,1
6,14 And b,14 And,Radial Velocity,2008,Okayama Astrophysical Observatory,1,Published Confirmed,0,185.84000,0.83,NaN,1525.5000,0.000,NaN,2.20,11.00,4813.0,1,1
12,14 Her b,14 Her,Radial Velocity,2002,W. M. Keck Observatory,1,Published Confirmed,0,1773.40002,2.93,NaN,1481.0878,0.370,NaN,0.90,0.93,5338.0,1,2
15,16 Cyg B b,16 Cyg B,Radial Velocity,1996,Multiple Observatories,1,Published Confirmed,0,798.50000,1.66,NaN,565.7374,0.680,NaN,1.08,1.13,5750.0,3,1


In [18]:
df_exoplanets.info()

<class 'pandas.DataFrame'>
Index: 5044 entries, 0 to 32551
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pl_name          5044 non-null   str    
 1   hostname         5044 non-null   str    
 2   discoverymethod  5044 non-null   str    
 3   disc_year        5044 non-null   int64  
 4   disc_facility    5044 non-null   str    
 5   default_flag     5044 non-null   int64  
 6   soltype          5044 non-null   str    
 7   pl_controv_flag  5044 non-null   int64  
 8   pl_orbper        4870 non-null   float64
 9   pl_orbsmax       2830 non-null   float64
 10  pl_rade          3898 non-null   float64
 11  pl_bmasse        2107 non-null   float64
 12  pl_orbeccen      1789 non-null   float64
 13  pl_eqt           952 non-null    float64
 14  st_mass          4284 non-null   float64
 15  st_rad           4585 non-null   float64
 16  st_teff          4678 non-null   float64
 17  sy_snum          5044 non-nul

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Carga tu dataset (reemplaza 'tu_dataset.csv' por el nombre de tu archivo real)
# df_exoplanets = pd.read_csv('tu_dataset.csv')

# 2. Definir variables predictoras y objetivo
features = ['disc_year', 'pl_orbper', 'pl_orbsmax', 'pl_rade', 'st_mass', 'st_rad', 'st_teff']
target = 'pl_eqt'

print(f"Total de registros iniciales: {len(df_exoplanets)}")

# 3. Eliminar TODAS las filas que contengan NaN en el target O en cualquiera de los features
columns_to_check = features + [target]
df_clean = df_exoplanets.dropna(subset=columns_to_check).copy()

print(f"Registros tras eliminar todas las filas con NaN: {len(df_clean)}")

if len(df_clean) > 0:
    X = df_clean[features]
    y = df_clean[target]

    # 4. Dividir en conjunto de entrenamiento (80%) y prueba (20%)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # --- APLICAR TRANSFORMACIÓN LOGARÍTMICA AL TARGET DE ENTRENAMIENTO ---
    # np.log1p(x) calcula log(1 + x), ideal para evitar problemas con ceros y comprimir la asimetría
    y_train_log = np.log1p(y_train)

    # 5. Entrenar el Random Forest
    rf_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )

    # Se entrena usando los valores transformados en logaritmo
    rf_model.fit(X_train, y_train_log)

    # 6. Predecir y revertir la transformación
    predictions_log = rf_model.predict(X_test)
    
    # np.expm1(x) revierte el log1p, devolviendo las predicciones a la escala original de masa
    predictions = np.expm1(predictions_log)

    print("\n--- RESULTADOS DE LAS PREDICCIONES (ESCALA REAL) ---")
    for real, pred in zip(y_test[:10], predictions[:10]): # Mostramos las primeras 10
        print(f"Temperatura Real: {real:.3f} | Temperatura Predicha: {pred:.3f}")

    # Las métricas se calculan comparando y_test (escala real) con predictions (escala real revertida)
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    print(f"\nError Cuadrático Medio (MSE): {mse:.3f}")
    print(f"Coeficiente de Determinación (R²): {r2:.3f}")
else:
    print("¡Atención! No quedaron registros después de limpiar todos los NaN. Revisa las columnas seleccionadas.")

Total de registros iniciales: 5044
Registros tras eliminar todas las filas con NaN: 766

--- RESULTADOS DE LAS PREDICCIONES (ESCALA REAL) ---
Temperatura Real: 243.000 | Temperatura Predicha: 368.961
Temperatura Real: 1526.000 | Temperatura Predicha: 1593.443
Temperatura Real: 634.000 | Temperatura Predicha: 619.223
Temperatura Real: 1136.000 | Temperatura Predicha: 1198.959
Temperatura Real: 226.000 | Temperatura Predicha: 299.801
Temperatura Real: 1871.000 | Temperatura Predicha: 1760.344
Temperatura Real: 208.000 | Temperatura Predicha: 284.977
Temperatura Real: 1193.000 | Temperatura Predicha: 1332.146
Temperatura Real: 2085.000 | Temperatura Predicha: 1967.129
Temperatura Real: 1649.000 | Temperatura Predicha: 1804.202

Error Cuadrático Medio (MSE): 21539.515
Coeficiente de Determinación (R²): 0.934
